# Classification des Genres Musicaux

**Projet** : Analyse et classification automatique de genres musicaux à partir de caractéristiques audio.

On utilise le dataset **GTZAN** qui contient :
- 1000 fichiers audio (format .wav)
- 10 genres : blues, classical, country, disco, hiphop, jazz, metal, pop, reggae, rock
- Chaque fichier dure 30 secondes

L'idée c'est d'extraire des features audio (MFCC, spectral centroid, etc.) puis d'entraîner des modèles de classification pour prédire le genre.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import os
import warnings
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')

# pour que les graphiques s'affichent dans le notebook
%matplotlib inline

print("Tout est importé !")

## 1. Chargement des données

On va parcourir le dossier `data/raw/` qui contient un sous-dossier par genre, chaque sous-dossier contient les fichiers `.wav`.

In [ ]:
# chemin vers les données
DATA_PATH = os.path.join('..', 'data', 'raw')

# on parcourt les dossiers pour récupérer les fichiers
fichiers = []
genres = []

for genre in os.listdir(DATA_PATH):
    dossier_genre = os.path.join(DATA_PATH, genre)
    if os.path.isdir(dossier_genre):
        for fichier in os.listdir(dossier_genre):
            if fichier.endswith('.wav'):
                fichiers.append(os.path.join(dossier_genre, fichier))
                genres.append(genre)

# on met tout dans un dataframe
df_audio = pd.DataFrame({'filepath': fichiers, 'genre': genres})
print(f"Nombre total de fichiers : {len(df_audio)}")
print()
print("Répartition par genre :")
print(df_audio['genre'].value_counts().sort_index())

## 2. Écouter un exemple

On charge un fichier audio pour voir à quoi ça ressemble. On utilise `librosa` pour lire le fichier.

In [ ]:
# on prend un fichier blues comme exemple
exemple = df_audio[df_audio['genre'] == 'blues'].iloc[0]['filepath']

# charger le fichier audio
y, sr = librosa.load(exemple, sr=22050)

print(f"Fichier : {exemple}")
print(f"Taux d'échantillonnage : {sr} Hz")
print(f"Durée : {len(y)/sr:.1f} secondes")
print(f"Nombre d'échantillons : {len(y)}")

# afficher la forme d'onde
plt.figure(figsize=(12, 3))
plt.plot(np.linspace(0, len(y)/sr, len(y)), y, color='steelblue', linewidth=0.5)
plt.xlabel('Temps (s)')
plt.ylabel('Amplitude')
plt.title(f'Forme d\'onde - Blues')
plt.tight_layout()
plt.show()

## 3. Visualisation audio

On va afficher 3 représentations du signal audio :
- **Forme d'onde** : le signal brut
- **Spectrogramme Mel** : représentation fréquentielle (plus utile pour l'analyse)
- **MFCC** : les coefficients cepstraux, très utilisés en classification audio

In [ ]:
# on reprend le même fichier blues
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# 1) forme d'onde
temps = np.linspace(0, len(y)/sr, len(y))
axes[0].plot(temps, y, color='steelblue', linewidth=0.5)
axes[0].set_title('Forme d\'onde')
axes[0].set_xlabel('Temps (s)')
axes[0].set_ylabel('Amplitude')

# 2) spectrogramme mel
mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
img = axes[1].imshow(mel_spec_db, aspect='auto', origin='lower', 
                      extent=[0, len(y)/sr, 0, sr/2])
axes[1].set_title('Spectrogramme Mel')
axes[1].set_xlabel('Temps (s)')
axes[1].set_ylabel('Fréquence (Hz)')
plt.colorbar(img, ax=axes[1], format='%+2.0f dB')

# 3) MFCC
mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
img2 = axes[2].imshow(mfccs, aspect='auto', origin='lower',
                       extent=[0, len(y)/sr, 0, 13])
axes[2].set_title('MFCC (13 coefficients)')
axes[2].set_xlabel('Temps (s)')
axes[2].set_ylabel('Coefficient')
plt.colorbar(img2, ax=axes[2])

plt.tight_layout()
plt.show()

## 4. Extraction des caractéristiques

On va extraire des features simples de chaque fichier audio :
- **13 MFCC** (moyenne sur le temps)
- **Spectral Centroid** (centre de gravité du spectre)
- **Zero Crossing Rate** (taux de passage par zéro)
- **Tempo** (BPM estimé)
- **Chroma** (moyenne globale des notes)

Ça nous donne environ 16 features par fichier, c'est suffisant pour une bonne classification.

In [ ]:
def extract_features(filepath):
    """extrait les features d'un fichier audio"""
    try:
        # on charge le fichier
        y, sr = librosa.load(filepath, sr=22050, duration=30)
        
        # mfcc - 13 coefficients, on prend la moyenne
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfcc_means = np.mean(mfccs, axis=1)
        
        # spectral centroid
        spectral_centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
        
        # zero crossing rate
        zcr = np.mean(librosa.feature.zero_crossing_rate(y))
        
        # tempo
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        if isinstance(tempo, np.ndarray):
            tempo = tempo[0]
        
        # chroma - moyenne globale
        chroma = np.mean(librosa.feature.chroma_stft(y=y, sr=sr))
        
        # on combine tout
        features = np.concatenate([
            mfcc_means,           # 13 features
            [spectral_centroid],  # 1 feature
            [zcr],                # 1 feature
            [tempo],              # 1 feature
            [chroma]              # 1 feature
        ])
        
        return features
    
    except Exception as e:
        print(f"Erreur avec {filepath}: {e}")
        return None

# test rapide sur un fichier
test_features = extract_features(exemple)
print(f"Nombre de features extraites : {len(test_features)}")
print(f"Aperçu : {test_features[:5]}")

In [ ]:
# noms des colonnes
col_names = [f'mfcc_{i+1}' for i in range(13)] + ['spectral_centroid', 'zcr', 'tempo', 'chroma']

# chemin pour sauvegarder les features (pour pas recalculer à chaque fois)
CSV_PATH = os.path.join('..', 'data', 'processed', 'features_simple.csv')

if os.path.exists(CSV_PATH):
    # si le fichier existe déjà on le charge directement
    print("Chargement des features depuis le CSV...")
    df = pd.read_csv(CSV_PATH)
    print(f"Features chargées : {df.shape}")
else:
    # sinon on extrait les features de tous les fichiers
    print("Extraction des features (ça peut prendre quelques minutes)...")
    
    all_features = []
    all_genres = []
    erreurs = 0
    
    for i, row in tqdm(df_audio.iterrows(), total=len(df_audio)):
        features = extract_features(row['filepath'])
        if features is not None:
            all_features.append(features)
            all_genres.append(row['genre'])
        else:
            erreurs += 1
    
    # on crée le dataframe
    df = pd.DataFrame(all_features, columns=col_names)
    df['genre'] = all_genres
    
    # on sauvegarde pour la prochaine fois
    os.makedirs(os.path.dirname(CSV_PATH), exist_ok=True)
    df.to_csv(CSV_PATH, index=False)
    
    print(f"\nTerminé ! {len(df)} fichiers traités, {erreurs} erreurs")

print()
df.head()

In [ ]:
# un petit describe pour voir les stats
df.describe()

## 5. Visualisation des features

On regarde comment les features se répartissent selon les genres pour voir si elles sont discriminantes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# boxplot du spectral centroid par genre
sns.boxplot(data=df, x='genre', y='spectral_centroid', ax=axes[0], palette='Set3')
axes[0].set_title('Spectral Centroid par genre')
axes[0].tick_params(axis='x', rotation=45)

# boxplot du tempo par genre
sns.boxplot(data=df, x='genre', y='tempo', ax=axes[1], palette='Set3')
axes[1].set_title('Tempo par genre')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# PCA pour visualiser les données en 2D
from sklearn.decomposition import PCA

features_cols = [c for c in df.columns if c != 'genre']
X_pca = StandardScaler().fit_transform(df[features_cols])
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_pca)

plt.figure(figsize=(10, 7))
for genre in df['genre'].unique():
    mask = df['genre'] == genre
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1], label=genre, alpha=0.6, s=30)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('PCA - Visualisation 2D des genres')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

print(f"Variance expliquée par les 2 composantes : {sum(pca.explained_variance_ratio_)*100:.1f}%")

## 6. Préparation des données

On sépare les features (X) et les labels (y), puis on fait un split train/test 80/20. On normalise les features avec StandardScaler.

In [ ]:
# séparer features et labels
features_cols = [c for c in df.columns if c != 'genre']
X = df[features_cols].values
y = df['genre'].values

# encoder les labels en nombres
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"Classes : {list(le.classes_)}")

# split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# normalisation
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"\nX_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

## 7. Entraînement des modèles

On teste 3 algorithmes de classification :
- **KNN** (K plus proches voisins) - simple et intuitif
- **SVM** (Support Vector Machine) - bon pour les petits datasets
- **Random Forest** - ensemble de arbres de décision

In [ ]:
# on définit les 3 modèles
modeles = {
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(kernel='rbf', C=10, probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

# on entraîne et évalue chaque modèle
resultats = {}

for nom, model in modeles.items():
    print(f"--- {nom} ---")
    
    # entraînement
    model.fit(X_train, y_train)
    
    # prédiction
    y_pred = model.predict(X_test)
    
    # accuracy
    acc = accuracy_score(y_test, y_pred)
    resultats[nom] = {'accuracy': acc, 'model': model, 'y_pred': y_pred}
    
    print(f"Accuracy : {acc:.4f} ({acc*100:.1f}%)")
    print()

# quel est le meilleur ?
meilleur = max(resultats, key=lambda x: resultats[x]['accuracy'])
print(f"==> Meilleur modèle : {meilleur} avec {resultats[meilleur]['accuracy']*100:.1f}%")

## 8. Résultats

On compare les modèles visuellement et on regarde en détail les performances du meilleur.

In [ ]:
# comparaison des accuracies
noms = list(resultats.keys())
accs = [resultats[n]['accuracy'] for n in noms]

plt.figure(figsize=(8, 5))
bars = plt.bar(noms, accs, color=['#5B9BD5', '#ED7D31', '#70AD47'])
plt.ylabel('Accuracy')
plt.title('Comparaison des modèles')
plt.ylim(0, 1)

# afficher les valeurs sur les barres
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{acc:.1%}', ha='center', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# matrice de confusion du meilleur modèle
y_pred_best = resultats[meilleur]['y_pred']
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.title(f'Matrice de confusion - {meilleur}')
plt.tight_layout()
plt.show()

In [ ]:
# rapport de classification détaillé
print(f"Rapport de classification - {meilleur}")
print("=" * 60)
print(classification_report(y_test, y_pred_best, target_names=le.classes_))

## 9. Test sur un nouveau fichier

On prend un fichier au hasard dans le jeu de test et on regarde ce que le modèle prédit.

In [ ]:
# on prend un fichier au hasard
np.random.seed(42)
idx = np.random.randint(0, len(df_audio))
fichier_test = df_audio.iloc[idx]

print(f"Fichier : {fichier_test['filepath']}")
print(f"Vrai genre : {fichier_test['genre']}")
print()

# extraire les features
features_test = extract_features(fichier_test['filepath'])

if features_test is not None:
    # normaliser
    features_test_scaled = scaler.transform(features_test.reshape(1, -1))
    
    # prédiction avec le meilleur modèle
    best_model = resultats[meilleur]['model']
    prediction = best_model.predict(features_test_scaled)
    genre_predit = le.inverse_transform(prediction)[0]
    
    print(f"Genre prédit : {genre_predit}")
    
    # probabilités (si le modèle le supporte)
    if hasattr(best_model, 'predict_proba'):
        probas = best_model.predict_proba(features_test_scaled)[0]
        
        # afficher les probabilités
        plt.figure(figsize=(10, 4))
        bars = plt.barh(le.classes_, probas, color='steelblue')
        plt.xlabel('Probabilité')
        plt.title(f'Prédiction pour : {os.path.basename(fichier_test["filepath"])}')
        
        # mettre en évidence le vrai genre
        for i, (cls, p) in enumerate(zip(le.classes_, probas)):
            if cls == fichier_test['genre']:
                bars[i].set_color('#70AD47')
            plt.text(p + 0.01, i, f'{p:.2%}', va='center')
        
        plt.tight_layout()
        plt.show()
        
    correct = "OUI" if genre_predit == fichier_test['genre'] else "NON"
    print(f"\nPrédiction correcte ? {correct}")

## Conclusion

Dans ce projet on a vu comment classifier automatiquement des genres musicaux :

1. **Exploration** : On a chargé et visualisé les fichiers audio du dataset GTZAN (formes d'onde, spectrogrammes, MFCC)

2. **Extraction de features** : On a extrait 16 caractéristiques simples de chaque fichier (MFCC, spectral centroid, ZCR, tempo, chroma)

3. **Classification** : On a testé 3 modèles (KNN, SVM, Random Forest) et comparé leurs performances

**Ce qu'on retient** :
- Les MFCC sont les features les plus importantes pour la classification audio
- SVM et Random Forest donnent les meilleurs résultats sur ce type de données
- Avec seulement 16 features simples, on arrive déjà à des résultats corrects
- Pour aller plus loin, on pourrait ajouter plus de features (delta MFCC, spectral bandwidth, etc.) ou utiliser des réseaux de neurones (CNN sur les spectrogrammes)